# Go Position Evaluation

**Numa Guiot and Enzo Picarel — ENSEIRB-MATMECA, 2026.**

This cleaned walkthrough uses the original CNN with a corrected symmetry-group split. See README.md and PROVENANCE.md for the distinction between coursework and later preparation. Run this notebook from the repository root.

In [ ]:
from pathlib import Path
import torch
import matplotlib.pyplot as plt
from go_model import load_samples, train, predict

torch.set_num_threads(2)
data_path = Path("data/samples-8x8.json.gz")
if not data_path.exists():
    raise FileNotFoundError("Place the course dataset in data/ as described in README.md")
samples = load_samples(data_path)
print(f"Loaded {len(samples):,} positions")

## Representation and evaluation

Each board has one plane for Black and one for White. The target is `black_wins / rollouts`. Identical boards and their eight symmetries stay together in one partition. Only training boards are augmented. This prevents symmetry leakage; it does not ensure game-level independence.

The CNN uses three padded convolutional blocks and a dense head with dropout. It omits side-to-move and game history; these are limitations rather than information the model can always infer.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model, history, metrics = train(samples, epochs=20, device=device)
metrics

## Learning curves

Compare training and validation losses and the constant-baseline MAE. These curves alone cannot establish generalization. No scores from the original leaky split are reused.

In [ ]:
plt.plot([r["epoch"] for r in history], [r["train_bce"] for r in history], label="Training")
plt.plot([r["epoch"] for r in history], [r["validation_bce"] for r in history], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Inference

The following is a format demonstration on a known position, not an independent test result. A probability estimates GNU Go rollout outcomes under the course data distribution.

In [ ]:
predict(model, samples[:1], device=device)

## Next experiments

Evaluate a game-level holdout where identifiers are available, encode side to move, compare with a simpler baseline, and measure symmetry consistency. These are future directions, not implemented results.